In [ ]:
# Conservative Data Cleaning Based on WER Cache

**Approach:** Only removes clearly broken samples, keeps structure intact.

## Key Improvements Over Original
- Data retention: ~85-90% instead of ~65%
- Preserves session/speaker structure (no concatenation)
- Evidence-based: Analyzes WER distribution first
- Conservative: Only removes catastrophic failures
- Reversible: Can tighten thresholds later

## Filtering Rules

REMOVE only if:
1. Missing audio file
2. Empty or invalid reference text
3. Severely distorted audio (clipping)
4. WER > 100% (complete transcription failures)
5. Fewer than 2 words
6. Duration > 30 seconds

KEEP:
- Samples with minor/moderate WER (10-50%)
- Samples with empty hypothesis (silence = valid child speech)
- Short utterances with ≥2 words

In [ ]:
# Import Required Libraries
import json
from pathlib import Path
from typing import List, Dict, Tuple
from collections import defaultdict

In [ ]:
# Function: Load original manifest
def load_original_manifest(manifest_path: Path) -> Dict[str, Dict]:
    """Load original manifest into dict keyed by audio_filepath."""
    manifest = {}
    with open(manifest_path) as f:
        for line in f:
            item = json.loads(line)
            manifest[item['audio_filepath']] = item
    return manifest

# Function: Load WER cache
def load_wer_cache(cache_path: Path) -> Dict[str, Dict]:
    """Load WER cache into dict keyed by audio_filepath."""
    cache = {}
    with open(cache_path) as f:
        for line in f:
            item = json.loads(line)
            cache[item['audio_filepath']] = item
    return cache

In [ ]:
# Function: Clean manifest with conservative rules
def clean_manifest(
    original_manifest: Dict[str, Dict],
    wer_cache: Dict[str, Dict],
    max_wer: float = 100.0,  # More conservative
    min_words: int = 2,  # More lenient
    max_duration: float = 30.0,
) -> Tuple[List[Dict], Dict[str, List[Dict]]]:
    """
    Apply conservative filtering rules.
    
    REMOVE only if:
    1. Missing audio file
    2. Empty or invalid reference text
    3. Severely distorted audio (clipping)
    4. WER > max_wer (default 100% = only remove complete garbage)
    5. Fewer than min_words
    6. Duration > max_duration
    
    KEEP everything else, including:
    - Samples with minor WER differences (10-50%)
    - Samples with empty Whisper hypothesis (silence is valid)
    - Short utterances with ≥2 words
    """
    kept = []
    removed = defaultdict(list)
    
    for audio_fp, orig_item in original_manifest.items():
        quality = wer_cache.get(audio_fp, {})
        
        # Get metadata
        duration = float(orig_item.get('duration', 0))
        ref_text = orig_item.get('text', '').strip()
        word_count = len(ref_text.split()) if ref_text else 0
        
        # Filtering rules
        reason = None
        
        # Rule 1: Missing audio
        if quality.get('missing_audio', False):
            reason = 'missing_audio'
        
        # Rule 2: Empty or whitespace-only reference
        elif not ref_text or ref_text.lower() in {
            '<discard>', '<no_signal>', '<silence>', '<noise>', '<non_speech>'
        }:
            reason = 'empty_or_nonspeech_reference'
        
        # Rule 3: Distorted audio (clipping)
        elif quality.get('distorted', False):
            reason = 'distorted_audio'
        
        # Rule 4: Catastrophic WER (complete transcription failure)
        elif quality.get('wer', 0) > max_wer:
            reason = f'wer_above_{max_wer}'
        
        # Rule 5: Too few words
        elif word_count < min_words:
            reason = 'too_few_words'
        
        # Rule 6: Too long
        elif duration > max_duration:
            reason = 'duration_exceeds_30s'
        
        # Keep this sample
        if reason is None:
            kept.append(orig_item)
        else:
            removed[reason].append({
                'audio_filepath': audio_fp,
                'duration': duration,
                'text': ref_text,
                'word_count': word_count,
                'wer': quality.get('wer'),
                'distorted': quality.get('distorted'),
                'hyp_norm': quality.get('hyp_norm', ''),
            })
    
    return kept, removed

In [ ]:
# Utility Functions: Save and Report
def save_manifest(items: List[Dict], output_path: Path):
    """Save cleaned manifest."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        for item in items:
            json.dump(item, f, ensure_ascii=False)
            f.write('\n')

def save_removal_report(removed: Dict[str, List[Dict]], output_path: Path):
    """Save detailed removal report."""
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, 'w') as f:
        json.dump(removed, f, indent=2, ensure_ascii=False)

def print_summary(split: str, original_count: int, kept: List[Dict], removed: Dict[str, List[Dict]]):
    """Print cleaning summary."""
    kept_dur = sum(float(x.get('duration', 0)) for x in kept)
    total_removed = sum(len(v) for v in removed.values())
    
    print(f"\n{'='*60}")
    print(f"SPLIT: {split}")
    print(f"{'='*60}")
    print(f"Original: {original_count} items")
    print(f"Kept: {len(kept)} items ({100*len(kept)/original_count:.1f}%, {kept_dur/3600:.2f}h)")
    print(f"Removed: {total_removed} items ({100*total_removed/original_count:.1f}%)")
    print(f"\nRemoval breakdown:")
    for reason, items in sorted(removed.items(), key=lambda x: len(x[1]), reverse=True):
        dur = sum(float(x.get('duration', 0)) for x in items)
        print(f"  {reason}: {len(items)} ({dur/3600:.2f}h)")

## Configuration

Set your data paths and cleaning parameters below:

In [ ]:
# Configuration Parameters
# ===========================

# Paths to original manifests (MODIFY THESE)
original_train = Path("/path/to/train.json")
original_val = Path("/path/to/val.json")
original_test = Path("/path/to/test.json")

# Directory containing WER cache files (train.jsonl, val.jsonl, test.jsonl)
wer_cache_dir = Path("/path/to/wer_cache")

# Output directory for cleaned manifests and reports
output_dir = Path("/lp-dev/amelia/data/myst/filtered_data")

# Cleaning parameters (conservative defaults)
max_wer = 100.0  # Only remove samples with WER > 100% (complete failures)
min_words = 2     # Keep samples with ≥2 words
max_duration = 50.0  # Remove samples longer than 50 seconds

print("Configuration loaded:")
print(f"  Original manifests: {original_train.parent}")
print(f"  WER cache dir: {wer_cache_dir}")
print(f"  Output dir: {output_dir}")
print(f"  Parameters: max_wer={max_wer}, min_words={min_words}, max_duration={max_duration}s")

## Execute Cleaning

Run the cleaning process on each data split (train, val, test):

In [ ]:
# Process all splits
results = {}

for split, orig_path in [
    ('train', original_train),
    ('val', original_val),
    ('test', original_test),
]:
    print(f"\n{'#'*70}")
    print(f"Processing {split.upper()}...")
    print(f"{'#'*70}")
    
    # Check if original manifest exists
    if not orig_path.exists():
        print(f"⚠️  WARNING: Original manifest not found at {orig_path}")
        continue
    
    # Check if WER cache exists
    cache_file = wer_cache_dir / f"{split}.jsonl"
    if not cache_file.exists():
        print(f"⚠️  WARNING: WER cache not found at {cache_file}")
        continue
    
    # Load data
    print(f"Loading {split} manifest...")
    original = load_original_manifest(orig_path)
    print(f"  ✓ Loaded {len(original)} items from {orig_path.name}")
    
    print(f"Loading WER cache...")
    wer_cache = load_wer_cache(cache_file)
    print(f"  ✓ Loaded {len(wer_cache)} cache entries")
    
    # Clean
    print(f"Applying conservative cleaning rules...")
    kept, removed = clean_manifest(
        original,
        wer_cache,
        max_wer=max_wer,
        min_words=min_words,
        max_duration=max_duration,
    )
    
    # Save cleaned manifest
    output_manifest = output_dir / f"{split}.json"
    save_manifest(kept, output_manifest)
    print(f"  ✓ Saved cleaned manifest to {output_manifest.name}")
    
    # Save removal report
    removal_report = output_dir / 'reports' / f"{split}_removed.json"
    save_removal_report(removed, removal_report)
    print(f"  ✓ Saved removal report to {removal_report.name}")
    
    # Print summary
    print_summary(split, len(original), kept, removed)
    
    # Store results
    results[split] = {
        'original': len(original),
        'kept': len(kept),
        'removed': sum(len(v) for v in removed.values()),
        'kept_duration': sum(float(x.get('duration', 0)) for x in kept),
        'removed_breakdown': {k: len(v) for k, v in removed.items()}
    }

print(f"\n{'='*70}")
print("CLEANING COMPLETE")
print(f"{'='*70}")

## Summary Statistics

Display overall cleaning statistics:

In [ ]:
# Display summary table
if results:
    print("\n" + "="*80)
    print("OVERALL CLEANING SUMMARY")
    print("="*80)
    print(f"{'Split':<10} {'Original':<12} {'Kept':<12} {'Removed':<12} {'Retention %':<15}")
    print("-"*80)
    
    total_original = 0
    total_kept = 0
    total_removed = 0
    
    for split in ['train', 'val', 'test']:
        if split in results:
            r = results[split]
            retention = (r['kept'] / r['original'] * 100) if r['original'] > 0 else 0
            print(f"{split:<10} {r['original']:<12} {r['kept']:<12} {r['removed']:<12} {retention:<14.1f}%")
            total_original += r['original']
            total_kept += r['kept']
            total_removed += r['removed']
    
    print("-"*80)
    total_retention = (total_kept / total_original * 100) if total_original > 0 else 0
    print(f"{'TOTAL':<10} {total_original:<12} {total_kept:<12} {total_removed:<12} {total_retention:<14.1f}%")
    print("="*80)
    
    print("\n✅ Cleaning complete! Check output directory for:")
    print(f"   - Cleaned manifests: {output_dir}/*.json")
    print(f"   - Removal reports: {output_dir}/reports/*.json")
else:
    print("⚠️  No splits were processed. Check your configuration paths.")